# Growing Neural Cellular Automata with 3D Voxels

This notebook trains a coarse 3D Neural Cellular Automaton (NCA) with an LPPN decoder, then exports a WebGPU-compatible voxel model for the interactive demo.

## Run order

1. Mount Google Drive.
2. Run setup, configure the model, and choose a target.
3. Train and inspect the generated volume.
4. Export the web model, make a verified Drive backup, then shut down the runtime.

The notebook includes:

- A configurable 3D growing NCA with stochastic updates, 3D Sobel/Laplacian perception, and alpha-based living masks.
- State-pool training with seed reinjection.
- A SIREN/LPPN decoder that maps upsampled cell state and local voxel coordinates to RGBA.
- Full-volume or sparse-target losses, depending on `CFG.use_sparse_target_loss`.
- Export files compatible with `demo/growing_voxels/`.

Sources used while adapting:  
Cells2Pixels repo: https://github.com/TheDevilWillBeBee/Cells2Pixels  
GNCA tutorial: https://quentinwach.com/blog/2025/06/10/gnca.html


## Setup and imports


In [ ]:
!pip install trimesh

import hashlib
import importlib
import json
import math
import os
import random
import shutil
import struct
import subprocess
import sys
import zipfile
from collections import OrderedDict
from contextlib import nullcontext
from dataclasses import asdict, dataclass, is_dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
# Import this explicitly before torch.optim. Some Colab runtimes can leave the
# private helper submodule unattached after package installation/lazy imports.
import torch._utils
import torch.nn as nn
import torch.nn.functional as F
import trimesh
from tqdm.auto import trange

try:
    import psutil
except ImportError:
    psutil = None

try:
    import gdown
except ImportError:
    gdown = None

try:
    from google.colab import drive, runtime
except ImportError:
    drive = None
    runtime = None


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(7)
if not hasattr(torch, '_utils'):
    raise RuntimeError(
        "PyTorch did not initialize correctly. Restart the Colab runtime, "
        "then run the notebook from the first cell."
    )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__)
print('device:', device)


## Mount Google Drive

Run this once at the start of a Colab session. It asks you to authorize Google Drive so the final backup cell can save the trained model before shutdown.


In [ ]:
#@title Mount Google Drive (run first)
if drive is None:
    raise RuntimeError('Google Drive mounting is available only in Google Colab.')

DRIVE_MOUNT = '/content/drive'
if not Path(DRIVE_MOUNT, 'MyDrive').exists():
    drive.mount(DRIVE_MOUNT, force_remount=False)
else:
    print(f'Google Drive is already mounted at {DRIVE_MOUNT}')


## Model and training configuration

Choose **Paper-style baseline** to use the original full-volume objective and coordinate encoding. Choose **Fine detail** for sparse 3D targets: it adds foreground/background-balanced loss, decoded Tversky shape supervision, straight-through living-gate gradients, and extra coordinate frequencies. Both modes export to the browser viewer.


In [ ]:
@dataclass
class CFG:
    target_inner_res: int = 128
    padding: int = 0
    scale_factor: int = 8
    channels: int = 48
    fc_dim: int = 256
    update_prob: float = 0.5
    seed_radius: int = 3
    pool_size: int = 128
    batch_size: int = 1
    virtual_batch_size: int = 1
    train_steps: int = 10000
    num_repetitions: int = 1
    step_range: tuple = (24, 56)
    inject_seed_interval: int = 2
    lr: float = 1e-3
    lr_decay_steps: tuple = (4000, 6000)
    lr_decay_gamma: float = 0.3
    overflow_weight: float = 100.0
    num_frequencies: int = 1
    training_mode: str = 'fine_detail'
    living_threshold: float = 0.1
    # Forward rendering remains hard-masked; this only supplies a smooth
    # backward path at fine living-mask boundaries during training.
    straight_through_living_gate: bool = True
    living_gate_temperature: float = 0.05
    # Opt-in for sparse 3D targets such as a pine tree. False preserves the
    # original Cells2Pixels-style full-volume L1/L2 objective exactly.
    use_sparse_target_loss: bool = True
    sparse_shape_weight: float = 1.0
    sparse_foreground_weight: float = 1.0
    sparse_background_weight: float = 1.0
    # Low-weight continuous NCA-alpha scaffold; final voxel losses remain primary.
    sparse_state_weight: float = 0.25
    precision: str = 'float16'

#@markdown Training objective: paper-style baseline or detail-focused 3D improvements.
TRAINING_MODE = 'Fine detail (recommended)' #@param ['Paper-style baseline', 'Fine detail (recommended)']

cfg = CFG()

def apply_training_mode(cfg, mode):
    if mode == 'Paper-style baseline':
        cfg.training_mode = 'paper_baseline'
        cfg.num_frequencies = 1
        cfg.straight_through_living_gate = False
        cfg.use_sparse_target_loss = False
    elif mode == 'Fine detail (recommended)':
        cfg.training_mode = 'fine_detail'
        cfg.num_frequencies = 3
        cfg.straight_through_living_gate = True
        cfg.use_sparse_target_loss = True
    else:
        raise ValueError(f'Unknown training mode: {mode}')
    return cfg

apply_training_mode(cfg, TRAINING_MODE)

if device.type == 'cpu':
    cfg.target_inner_res = 128
    cfg.padding = 2
    cfg.channels = 48
    cfg.fc_dim =256
    cfg.pool_size = 8
    cfg.batch_size = 1
    cfg.virtual_batch_size = 1
    cfg.train_steps = 250
    cfg.num_repetitions = 1
    cfg.step_range = (32, 64)
    cfg.inject_seed_interval = 2
    cfg.overflow_weight = 100.0
    cfg.precision = 'float16'



assert (cfg.target_inner_res + 2 * cfg.padding) % cfg.scale_factor == 0
print(f'training mode: {TRAINING_MODE} | sparse loss: {cfg.use_sparse_target_loss} | frequencies: {cfg.num_frequencies}')
cfg


## Target volumes and loaders

The active workflow below trains on a dense `.vox` or `.npy` target. The optional mesh workflow supports a Cells2Pixels-style `.obj` mesh plus `.vol` solid texture.

The training tensor is always dense RGBA `[1, D, H, W, 4]` in `zyx` order.


In [ ]:
# Optional mesh + texture assets for the OBJ + VOL workflow.
# Leave this False for dense .vox / .npy training.
DOWNLOAD_OPTIONAL_MESH_DATASETS = False #@param {type:"boolean"}

if DOWNLOAD_OPTIONAL_MESH_DATASETS:
    if gdown is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])
        gdown = __import__('gdown')

    def download_zip_and_extract(url, zip_path, output_dir, dataset_name):
        if os.path.exists(output_dir):
            print(f'{dataset_name} is already downloaded.')
            return
        os.makedirs(os.path.dirname(zip_path), exist_ok=True)
        gdown.download(url, zip_path, quiet=False)
        print(f'Unzipping {dataset_name}...')
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(output_dir)
        os.remove(zip_path)
        print(f'Finished {dataset_name}.')

    download_zip_and_extract(
        url='https://drive.google.com/uc?id=136uliL3tcQinNXg3LXcJB4_Qf-HXg2A3',
        zip_path='data/meshes.zip',
        output_dir='data/meshes',
        dataset_name='Mesh dataset',
    )
    download_zip_and_extract(
        url='https://drive.google.com/uc?id=1Xa5RXDs_sEW7HihdMwG3duG-vViraY0f',
        zip_path='data/solid_textures.zip',
        output_dir='data/solid_textures',
        dataset_name='Solid texture dataset',
    )
else:
    print('Skipping optional OBJ + VOL datasets.')


In [ ]:
def _center_crop_or_pad(arr, shape):
    out = arr
    for axis, target in enumerate(shape):
        size = out.shape[axis]
        if size > target:
            start = (size - target) // 2
            idx = [slice(None)] * out.ndim
            idx[axis] = slice(start, start + target)
            out = out[tuple(idx)]
    pads = []
    for axis, target in enumerate(shape):
        size = out.shape[axis]
        before = max((target - size) // 2, 0)
        after = max(target - size - before, 0)
        pads.append((before, after))
    return np.pad(out, pads + [(0, 0)] * (out.ndim - len(pads)), mode='constant')


def normalize_rgba_array(arr, coord_order='zyx'):
    arr = np.asarray(arr)

    # Handle sparse list format [N, 6] -> [z, y, x, r, g, b]
    if arr.ndim == 2 and arr.shape[1] == 6:
        coords = arr[:, :3].astype(int)
        colors = arr[:, 3:].astype(np.float32)
        if colors.max() > 1.0:
            colors /= 255.0

        mins = coords.min(0)
        coords -= mins
        maxs = coords.max(0)

        dense = np.zeros((maxs[0]+1, maxs[1]+1, maxs[2]+1, 4), dtype=np.float32)
        for i in range(len(coords)):
            z, y, x = coords[i]
            dense[z, y, x, :3] = colors[i]
            dense[z, y, x, 3] = 1.0
        arr = dense

    if arr.ndim not in (3, 4):
        raise ValueError(f'Expected [D,H,W], [D,H,W,C], [X,Y,Z], or [X,Y,Z,C], got {arr.shape}')

    if coord_order.lower() == 'xyz':
        axes = (2, 1, 0) if arr.ndim == 3 else (2, 1, 0, 3)
        arr = np.transpose(arr, axes)
    elif coord_order.lower() != 'zyx':
        raise ValueError("coord_order must be 'zyx' or 'xyz'")

    arr = arr.astype(np.float32, copy=False)
    if arr.size and arr.max() > 1.0:
        arr = arr / 255.0

    if arr.ndim == 3:
        a = np.clip(arr[..., None], 0, 1)
        return np.concatenate([a, a, a, a], axis=-1).astype(np.float32)

    c = arr.shape[-1]
    if c == 1:
        a = np.clip(arr, 0, 1)
        return np.concatenate([a, a, a, a], axis=-1).astype(np.float32)
    if c == 3:
        rgb = np.clip(arr, 0, 1)
        a = (rgb.max(axis=-1, keepdims=True) > 0).astype(np.float32)
        return np.concatenate([rgb * a, a], axis=-1).astype(np.float32)
    if c >= 4:
        rgba = np.clip(arr[..., :4], 0, 1)
        rgba[..., :3] *= rgba[..., 3:4]
        return rgba.astype(np.float32)
    raise ValueError(f'Unsupported channel count: {c}')


def _load_midvoxio(auto_install=True):
    try:
        return importlib.import_module('midvoxio.voxio').vox_to_arr
    except Exception:
        if not auto_install:
            return None
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/midstreeeam/MidVoxIO.git'])
        return importlib.import_module('midvoxio.voxio').vox_to_arr


def read_magica_vox(path):
    path = Path(path)
    with open(path, 'rb') as f:
        data = f.read()
    if data[:4] != b'VOX ':
        raise ValueError('Not a MagicaVoxel .vox file')

    offset = 8
    size_xyz = None
    xyz_i = []
    palette = np.zeros((256, 4), dtype=np.uint8)
    palette[:, :3] = 255
    palette[:, 3] = 255

    while offset + 12 <= len(data):
        chunk_id = data[offset:offset+4]
        content_size = int.from_bytes(data[offset+4:offset+8], 'little')
        children_size = int.from_bytes(data[offset+8:offset+12], 'little')
        content = data[offset+12:offset+12+content_size]
        offset = offset + 12 + content_size

        if chunk_id == b'SIZE' and len(content) >= 12:
            size_xyz = tuple(int.from_bytes(content[i:i+4], 'little') for i in (0, 4, 8))
        elif chunk_id == b'XYZI' and len(content) >= 4:
            n = int.from_bytes(content[:4], 'little')
            pts = np.frombuffer(content[4:4+n*4], dtype=np.uint8).reshape(-1, 4)
            xyz_i.extend([tuple(map(int, p)) for p in pts])
        elif chunk_id == b'RGBA' and len(content) >= 1024:
            palette = np.frombuffer(content[:1024], dtype=np.uint8).reshape(256, 4).copy()
        offset += children_size

    if size_xyz is None:
        if xyz_i:
            pts = np.array(xyz_i, dtype=np.int32)
            size_xyz = tuple((pts[:, :3].max(axis=0) + 1).tolist())
        else:
            raise ValueError('No SIZE/XYZI chunks found in .vox')
    return xyz_i, palette, size_xyz


def load_voxel_file(path, coord_order='zyx', auto_install_midvoxio=True):
    """Load .npy or .vox to dense float32 [D,H,W,4]."""
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == '.npy':
        return normalize_rgba_array(np.load(path, allow_pickle=True), coord_order=coord_order)

    if suffix == '.vox':
        vox_to_arr = _load_midvoxio(auto_install=auto_install_midvoxio)
        if vox_to_arr is not None:
            try:
                arr = vox_to_arr(str(path), -1)
                return normalize_rgba_array(arr, coord_order='xyz')
            except Exception as exc:
                print(f"midvoxio failed; falling back to minimal parser. ({exc})")
        xyz_i, palette, size_xyz = read_magica_vox(path)
        xs, ys, zs = [int(v) for v in size_xyz]
        volume = np.zeros((zs, ys, xs, 4), dtype=np.float32)
        for x, y, z, ci in xyz_i:
            if 0 <= x < xs and 0 <= y < ys and 0 <= z < zs:
                color = palette[max(0, int(ci) - 1)].astype(np.float32) / 255.0
                color[3] = 1.0
                volume[z, y, x] = color
        return normalize_rgba_array(volume, coord_order='zyx')

    raise ValueError(f'Unsupported voxel file: {path.suffix}')


def _pad_rgba_volume(rgba, padding):
    if isinstance(padding, int):
        pads = [(padding, padding)] * 3
    else:
        pads = list(padding)
        if len(pads) == 3:
            pads = [(int(p), int(p)) for p in pads]
        elif len(pads) == 6:
            pads = [(int(pads[0]), int(pads[1])), (int(pads[2]), int(pads[3])), (int(pads[4]), int(pads[5]))]
        else:
            raise ValueError('padding must be an int, 3-tuple, or 6-tuple')
    return np.pad(rgba, pads + [(0, 0)], mode='constant')


def _pad_to_divisible(rgba, divisor):
    if divisor is None or int(divisor) <= 1:
        return rgba, (0, 0, 0)
    divisor = int(divisor)
    extras = tuple((divisor - (s % divisor)) % divisor for s in rgba.shape[:3])
    if not any(extras):
        return rgba, extras
    pads = [(e // 2, e - e // 2) for e in extras]
    return np.pad(rgba, pads + [(0, 0)], mode='constant'), extras


def dense_file_to_target(path, res=None, padding=8, coord_order='zyx', device=device, make_cubic=True, ensure_divisible_by=1):
    rgba = load_voxel_file(path, coord_order=coord_order)
    # Center the active voxels within the bounding box
    alpha = rgba[..., 3]
    nonzero = np.argwhere(alpha > 0.05)
    if len(nonzero) > 0:
        min_c = nonzero.min(axis=0)
        max_c = nonzero.max(axis=0)
        cropped = rgba[min_c[0]:max_c[0]+1, min_c[1]:max_c[1]+1, min_c[2]:max_c[2]+1]
        rgba = _center_crop_or_pad(cropped, rgba.shape[:3])
    original_shape = tuple(int(v) for v in rgba.shape[:3])

    if res is not None:
        res = int(res)
        rgba = _center_crop_or_pad(rgba, (res, res, res))
    elif make_cubic:
        cube_size = int(max(rgba.shape[:3]))
        rgba = _center_crop_or_pad(rgba, (cube_size, cube_size, cube_size))

    pre_pad_shape = tuple(int(v) for v in rgba.shape[:3])
    rgba = _pad_rgba_volume(rgba, int(padding))
    rgba, extra = _pad_to_divisible(rgba, ensure_divisible_by)
    final_shape = tuple(int(v) for v in rgba.shape[:3])

    print('loaded dense target:', path)
    print('  original:', original_shape)
    print('  before padding:', pre_pad_shape)
    print('  padding each side:', int(padding))
    if any(extra):
        print('  extra empty padding for scale divisibility:', extra)
    print('  final decoded target:', final_shape)

    return torch.tensor(rgba[None], dtype=torch.float32, device=device)


def read_cells2pixels_vol(filepath):
    with open(filepath, 'rb') as f:
        header = f.read(4096)
        magic, version, tex_name, wrap, vol_size, num_channels, bytes_per_channel = struct.Struct('4si256s?iii').unpack_from(header)
        if magic != b'VOLU' or version != 4 or bytes_per_channel != 1:
            raise ValueError('Unsupported .vol file')
        n = vol_size * vol_size * vol_size * num_channels
        data = np.frombuffer(f.read(n), dtype=np.uint8)
    return data.reshape((vol_size, vol_size, vol_size, num_channels))


def load_mesh_solid_texture_target(mesh_path, vol_path, res=32, padding=8, device=device, ensure_divisible_by=1):
    mesh = trimesh.load(mesh_path, force='mesh')
    mesh.apply_translation(-mesh.centroid)
    mesh.apply_scale(1.0 / max(mesh.extents))
    occupancy = mesh.voxelized(pitch=1.0 / (res - 1)).fill().matrix.astype(np.float32)
    occupancy = _center_crop_or_pad(occupancy[..., None], (res, res, res))[..., 0]

    solid = read_cells2pixels_vol(vol_path).astype(np.float32) / 255.0
    reps = [res // solid.shape[i] + int(res % solid.shape[i] != 0) for i in range(3)]
    color = np.tile(solid, (*reps, 1))[:res, :res, :res, :3]
    rgba = np.concatenate([color * occupancy[..., None], occupancy[..., None]], axis=-1)
    rgba = np.pad(rgba, ((padding, padding), (padding, padding), (padding, padding), (0, 0)))
    rgba, extra = _pad_to_divisible(rgba, ensure_divisible_by)
    return torch.tensor(rgba[None], dtype=torch.float32, device=device)


def load_obj_target(mesh_path, res=64, padding=8, color=[0.5, 0.5, 0.5], device=device, ensure_divisible_by=1):
    mesh = trimesh.load(mesh_path, force='mesh')
    mesh.apply_translation(-mesh.centroid)
    mesh.apply_scale(1.0 / max(mesh.extents))
    voxel_grid = mesh.voxelized(pitch=1.0 / (res - 1))
    occupancy = voxel_grid.fill().matrix.astype(np.float32)
    occupancy = _center_crop_or_pad(occupancy[..., None], (res, res, res))[..., 0]
    rgb = np.zeros((*occupancy.shape, 3), dtype=np.float32)
    rgb[occupancy > 0.5] = color
    rgba = np.concatenate([rgb, occupancy[..., None]], axis=-1)
    rgba = _pad_rgba_volume(rgba, int(padding))
    rgba, extra = _pad_to_divisible(rgba, ensure_divisible_by)
    return torch.tensor(rgba[None], dtype=torch.float32, device=device)


## NCA, LPPN, loss, and training utilities


In [ ]:

def depthwise_conv3d(x, filters, padding='circular'):
    b, c, d, h, w = x.shape
    y = x.reshape(b * c, 1, d, h, w)
    y = F.pad(y, (1, 1, 1, 1, 1, 1), mode=padding)
    y = F.conv3d(y, filters[:, None])
    return y.reshape(b, c * filters.shape[0], d, h, w)


class VNCA3D(nn.Module):
    def __init__(self, channels=16, fc_dim=128, update_prob=0.5, padding='circular', precision=torch.float32):
        super().__init__()
        self.channels = channels
        self.update_prob = update_prob
        self.padding = padding
        self.perception_kernels = 5
        self.precision = precision
        self.w1 = nn.Conv3d(channels * self.perception_kernels, fc_dim, 1)
        self.w2 = nn.Conv3d(fc_dim, channels, 1, bias=False)
        nn.init.xavier_normal_(self.w1.weight, gain=0.1)
        nn.init.xavier_normal_(self.w2.weight, gain=0.1)
        self.register_buffer('filters', self._filters(precision))

    @staticmethod
    def _filters(precision):
        d1 = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=precision)
        d2 = torch.tensor([[-2., 0., 2.], [-4., 0., 4.], [-2., 0., 2.]], dtype=precision)
        sobel_z = torch.stack([d1, d2, d1]) / 2.0
        sobel_y = sobel_z.permute(0, 2, 1)
        sobel_x = sobel_z.permute(2, 1, 0)
        lap1 = torch.tensor([[2., 3., 2.], [3., 6., 3.], [2., 3., 2.]], dtype=precision)
        lap2 = torch.tensor([[3., 6., 3.], [6., -88., 6.], [3., 6., 3.]], dtype=precision)
        lap = torch.stack([lap1, lap2, lap1]) / 8.0
        ident = torch.zeros(3, 3, 3, dtype=precision)
        ident[1, 1, 1] = 1.0
        return torch.stack([ident, sobel_x, sobel_y, sobel_z, lap])

    def forward(self, s):
        z = depthwise_conv3d(s, self.filters.to(dtype=s.dtype), self.padding)
        ds = self.w2(F.relu(self.w1(z)))
        if self.update_prob < 1.0:
            mask = (torch.rand(s.shape[0], 1, s.shape[2], s.shape[3], s.shape[4], device=s.device, dtype=self.precision) < self.update_prob).to(s.dtype)
            ds = ds * mask
        return s + ds, z

    def seed(self, n, d, h, w):
        return torch.zeros(n, self.channels, d, h, w, device=next(self.parameters()).device, dtype=self.precision)


class GrowingVNCA3D(VNCA3D):
    def __init__(self, seed_radius=1, living_threshold=0.1,
                 straight_through_living_gate=True, living_gate_temperature=0.05, **kwargs):
        super().__init__(**kwargs)
        self.seed_radius = seed_radius
        self.living_threshold = float(living_threshold)
        self.straight_through_living_gate = bool(straight_through_living_gate)
        self.living_gate_temperature = float(living_gate_temperature)

    @staticmethod
    def _pooled_alpha(s):
        return F.max_pool3d(s[:, 3:4], 3, stride=1, padding=1)

    def living_mask(self, s):
        return self._pooled_alpha(s) > self.living_threshold

    def render_living_gate(self, s):
        """Hard forward gate with an optional smooth training-time backward path."""
        pooled_alpha = self._pooled_alpha(s)
        hard = (pooled_alpha > self.living_threshold).to(s.dtype)
        if self.straight_through_living_gate and torch.is_grad_enabled():
            temperature = max(self.living_gate_temperature, 1e-4)
            soft = torch.sigmoid((pooled_alpha - self.living_threshold) / temperature)
            return hard.detach() - soft.detach() + soft
        return hard

    def forward(self, s):
        pre = self.living_mask(s)
        s, z = super().forward(s)
        post = self.living_mask(s)
        return s * (pre & post).to(self.precision), z

    def seed(self, n, d, h, w):
        s = super().seed(n, d, h, w)
        r = self.seed_radius - 1
        c = (d // 2, h // 2, w // 2)
        s[:, 3:, c[0]-r:c[0]+r+1, c[1]-r:c[1]+r+1, c[2]-r:c[2]+r+1] = 1.0
        return s


In [ ]:
class SineLayer(nn.Module):
    def __init__(self, in_features, out_features, is_first=False, omega_0=10.0, activation='sin', fx='linear'):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.fx = fx
        self.linear = nn.Linear(in_features, out_features)
        self.activation = torch.sin if activation == 'sin' else getattr(F, activation)
        self.init_weights(in_features)

    def init_weights(self, in_features):
        with torch.no_grad():
            bound = 1 / in_features if self.is_first else math.sqrt(6 / in_features) / self.omega_0
            self.linear.weight.uniform_(-bound, bound)

    def forward(self, x):
        x = self.linear(x)
        if self.fx == 'finer':
            x = x * (1.0 + torch.abs(x))
        elif self.fx == 'sinh':
            x = torch.sinh(2.0 * x)
        return self.activation(self.omega_0 * x)


class SirenDecoder(nn.Module):
    def __init__(self, in_features, coord_dim=3, hidden_features=32, hidden_layers=2, out_features=4,
                 num_frequencies=1, first_omega_0=10.0, hidden_omega_0=10.0, fx='linear', activation='sin'):
        super().__init__()
        self.num_frequencies = num_frequencies
        enc_coord_dim = coord_dim * max(1, 2 * num_frequencies)
        layers = [SineLayer(in_features + enc_coord_dim, hidden_features, is_first=True,
                            omega_0=first_omega_0, activation=activation, fx=fx)]
        for _ in range(hidden_layers):
            layers.append(SineLayer(hidden_features, hidden_features, omega_0=hidden_omega_0,
                                    activation=activation, fx=fx))
        final = nn.Linear(hidden_features, out_features)
        with torch.no_grad():
            final.weight.uniform_(-math.sqrt(6 / hidden_features) / hidden_omega_0,
                                  math.sqrt(6 / hidden_features) / hidden_omega_0)
        layers.append(final)
        self.net = nn.Sequential(*layers)

    def encode_coords(self, coords):
        if self.num_frequencies <= 0:
            return coords
        freqs = torch.arange(1, self.num_frequencies + 1, device=coords.device, dtype=coords.dtype)
        a = coords[..., None, :] * math.pi * freqs[:, None]
        return torch.cat([torch.sin(a).flatten(-2), torch.cos(a).flatten(-2)], dim=-1)

    def forward(self, cell_states, coords):
        x = torch.cat([self.encode_coords(coords), cell_states], dim=-1)
        return self.net(x)

In [ ]:

def precision_from_cfg(cfg):
    if device.type == 'cuda' and getattr(cfg, 'precision', 'float32') == 'float16':
        return torch.float16
    return torch.float32


def autocast_context(run_device, precision):
    enabled = run_device.type == 'cuda' and precision == torch.float16
    return torch.autocast(device_type=run_device.type, dtype=precision) if enabled else nullcontext()


def make_grad_scaler(run_device, precision):
    enabled = run_device.type == 'cuda' and precision == torch.float16
    return torch.amp.GradScaler(run_device.type, enabled=enabled)


def optimizer_scheduler_step(opt, sched, scaler, precision):
    if precision == torch.float16:
        scale_before = scaler.get_scale()
        scaler.step(opt)
        scaler.update()
        stepped = scaler.get_scale() >= scale_before
    else:
        opt.step()
        stepped = True
    if stepped:
        sched.step()


def local_subcell_coords(scale_factor, device, dtype=torch.float32):
    t = (torch.arange(scale_factor, device=device, dtype=dtype) / scale_factor - 0.5 + 0.5 / scale_factor) * 2.0
    z, y, x = torch.meshgrid(t, t, t, indexing='ij')
    return torch.stack([z, y, x], dim=-1)


def decode_voxel(model, siren, state, z, grid_size, scale_factor, precision):
    x_render = state.float()
    x_pad = F.pad(x_render, (1, 1, 1, 1, 1, 1), mode='circular')
    x_up = F.interpolate(x_pad, scale_factor=scale_factor, mode='trilinear', align_corners=False)
    x_up = x_up[:, :, scale_factor:-scale_factor, scale_factor:-scale_factor, scale_factor:-scale_factor]
    b = x_up.shape[0]
    coords = local_subcell_coords(scale_factor, x_up.device, torch.float32)[None].repeat(b, *grid_size, 1)
    cells = x_up.permute(0, 2, 3, 4, 1)
    living = model.render_living_gate(x_up).permute(0, 2, 3, 4, 1)
    with autocast_context(x_up.device, precision):
        generated = siren(cells, coords)
    return (generated * living).float(), x_up.float()


def psnr(x, y):
    mse = ((x.clamp(0, 1) - y.clamp(0, 1)) ** 2).mean()
    return -10.0 * torch.log10(mse + 1e-8)


def voxel_morphogenesis_loss(generated, target, state, upsampled_state, overflow_weight=100.0,
                              use_sparse_target_loss=False, sparse_shape_weight=1.0,
                              sparse_foreground_weight=1.0, sparse_background_weight=1.0,
                              sparse_state_weight=0.25):
    alpha_state = upsampled_state[:, 3:4].permute(0, 2, 3, 4, 1) * 2.0
    rgba_error = generated - target
    rgba_l2 = (rgba_error ** 2).mean()
    rgba_l1 = rgba_error.abs().mean()
    occ_error = alpha_state - target[..., 3:4]
    occ_l2 = (occ_error ** 2).mean()
    occ_l1 = occ_error.abs().mean()
    overflow_state = state.float()
    overflow = (overflow_state - overflow_state.clamp(-1.0, 1.0)).abs().mean()

    if use_sparse_target_loss:
        eps = 1e-6
        target_alpha = target[..., 3:4].clamp(0, 1)
        foreground = (target_alpha > 0.05).float()
        background = 1.0 - foreground

        def masked_l1(error, mask):
            return (error.abs() * mask).sum() / (mask.sum() * error.shape[-1] + eps)

        def masked_l2(error, mask):
            return ((error ** 2) * mask).sum() / (mask.sum() * error.shape[-1] + eps)

        # The decoded RGBA field is the visible geometry. Supervise both target
        # foreground and target air separately so a live coarse envelope cannot
        # hide arbitrary opaque LPPN voxels in its interior.
        fg_decoded_l1 = masked_l1(rgba_error, foreground)
        fg_decoded_l2 = masked_l2(rgba_error, foreground)
        bg_decoded_l1 = masked_l1(rgba_error, background)
        bg_decoded_l2 = masked_l2(rgba_error, background)

        # Keep continuous NCA alpha as a low-weight growth scaffold. Do not clamp
        # it here: clamp would remove gradients once alpha reaches its bounds.
        fg_state_alpha = masked_l1(occ_error, foreground)
        bg_state_alpha = masked_l1(occ_error, background)

        # Tversky must describe the rendered voxels, not the interpolated coarse
        # alpha. Sigmoid supplies a differentiable bounded occupancy probability.
        decoded_alpha_prob = torch.sigmoid((generated[..., 3:4] - 0.5) * 8.0)
        tp = (decoded_alpha_prob * target_alpha).sum()
        fp = (decoded_alpha_prob * (1.0 - target_alpha)).sum()
        fn = ((1.0 - decoded_alpha_prob) * target_alpha).sum()
        tversky = 1.0 - (tp + eps) / (tp + 0.3 * fp + 0.7 * fn + eps)

        foreground_loss = fg_decoded_l1 + fg_decoded_l2 + sparse_state_weight * fg_state_alpha
        background_loss = bg_decoded_l1 + bg_decoded_l2 + sparse_state_weight * bg_state_alpha
        loss = (sparse_foreground_weight * foreground_loss
                + sparse_background_weight * background_loss
                + sparse_shape_weight * tversky
                + overflow_weight * overflow)
    else:
        loss = rgba_l2 + rgba_l1 + occ_l2 + occ_l1 + overflow_weight * overflow

    return loss, {
        'loss': float(loss.detach()),
        'rgba_l2': float(rgba_l2.detach()),
        'rgba_l1': float(rgba_l1.detach()),
        'occ_l2': float(occ_l2.detach()),
        'occ_l1': float(occ_l1.detach()),
        'sparse_loss_enabled': bool(use_sparse_target_loss),
        'fg_decoded_l1': float(fg_decoded_l1.detach()) if use_sparse_target_loss else 0.0,
        'fg_decoded_l2': float(fg_decoded_l2.detach()) if use_sparse_target_loss else 0.0,
        'bg_decoded_l1': float(bg_decoded_l1.detach()) if use_sparse_target_loss else 0.0,
        'bg_decoded_l2': float(bg_decoded_l2.detach()) if use_sparse_target_loss else 0.0,
        'fg_state_alpha': float(fg_state_alpha.detach()) if use_sparse_target_loss else 0.0,
        'bg_state_alpha': float(bg_state_alpha.detach()) if use_sparse_target_loss else 0.0,
        'tversky': float(tversky.detach()) if use_sparse_target_loss else 0.0,
        'overflow': float(overflow.detach()),
        'psnr': float(psnr(generated, target).detach()),
        'alpha_max': float(alpha_state.detach().max()),
    }

def normalize_model_grads(model):
    for p in model.parameters():
        if p.grad is not None:
            p.grad /= p.grad.norm() + 1e-8

In [ ]:
def rgba_on_white(x):
    x = x.detach().float().cpu().numpy()
    rgb = np.clip(x[..., :3], 0, 1)
    a = np.clip(x[..., 3:4], 0, 1)
    return np.clip(rgb * a + (1.0 - a), 0, 1)


def show_slices(generated, target, title='generated vs target'):
    gen = generated[0]
    tgt = target[0]
    d, h, w = gen.shape[:3]
    cuts = [(d // 2, 'YZ'), (h // 2, 'XZ'), (w // 2, 'XY')]
    fig, axes = plt.subplots(2, 3, figsize=(10, 6))
    for j, (idx, name) in enumerate(cuts):
        gs = gen[idx] if j == 0 else gen[:, idx] if j == 1 else gen[:, :, idx]
        ts = tgt[idx] if j == 0 else tgt[:, idx] if j == 1 else tgt[:, :, idx]
        axes[0, j].imshow(rgba_on_white(gs)); axes[0, j].set_title(f'gen {name}')
        axes[1, j].imshow(rgba_on_white(ts)); axes[1, j].set_title(f'target {name}')
        axes[0, j].axis('off'); axes[1, j].axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_history(history):
    if not history:
        return
    steps = [h['step'] for h in history]
    plt.figure(figsize=(7, 3))
    plt.plot(steps, [h['loss'] for h in history], label='loss')
    if history[0].get('sparse_loss_enabled'):
        plt.plot(steps, [h['fg_decoded_l1'] for h in history], label='foreground decoded L1')
        plt.plot(steps, [h['bg_decoded_l1'] for h in history], label='background decoded L1')
        plt.plot(steps, [h['tversky'] for h in history], label='decoded Tversky')
    else:
        plt.plot(steps, [h['occ_l2'] for h in history], label='occupancy L2')
    plt.yscale('log')
    plt.legend()
    plt.xlabel('step')
    plt.show()


def voxel_scatter(rgba, threshold=0.5, max_points=12000):
    v = rgba[0].detach().float().cpu()
    alpha = v[..., 3].clamp(0, 1)
    pts = torch.nonzero(alpha > threshold)
    if len(pts) == 0:
        print('no occupied voxels above threshold')
        print('alpha min/max/mean:', float(alpha.min()), float(alpha.max()), float(alpha.mean()))
        return
    if len(pts) > max_points:
        pts = pts[torch.randperm(len(pts))[:max_points]]
    colors = v[pts[:, 0], pts[:, 1], pts[:, 2], :3].clamp(0, 1).numpy()
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(pts[:, 2], pts[:, 1], pts[:, 0], s=2, c=colors)
    ax.set_axis_off()
    plt.show()

In [ ]:
def make_models(cfg):
    precision = precision_from_cfg(cfg)
    model = GrowingVNCA3D(
        channels=cfg.channels,
        fc_dim=cfg.fc_dim,
        update_prob=cfg.update_prob,
        seed_radius=cfg.seed_radius,
        living_threshold=cfg.living_threshold,
        straight_through_living_gate=cfg.straight_through_living_gate,
        living_gate_temperature=cfg.living_gate_temperature,
        precision=precision,
    ).to(device)
    siren = SirenDecoder(
        cfg.channels,
        hidden_features=32,
        hidden_layers=2,
        out_features=4,
        num_frequencies=cfg.num_frequencies,
        first_omega_0=10.0,
        hidden_omega_0=10.0,
    ).to(device)
    return model, siren, precision


def prune_dead_pool(pool, model, grid_size):
    """Replace states with no cells that satisfy the model's actual living rule."""
    with torch.no_grad():
        alive_per_sample = model.living_mask(pool).flatten(1).any(dim=1)
        dead_mask = ~alive_per_sample
        n_dead = int(dead_mask.sum().item())
        if n_dead > 0:
            fresh = model.seed(n_dead, *grid_size)
            pool[dead_mask] = fresh
        return n_dead


def train_voxel_gnca(target, cfg):
    model, siren, precision = make_models(cfg)
    grid_size = tuple(s // cfg.scale_factor for s in target.shape[1:4])
    history = []
    accumulation_steps = max(1, math.ceil(cfg.virtual_batch_size / cfg.batch_size))

    for rep in range(cfg.num_repetitions):
        pool = model.seed(cfg.pool_size, *grid_size)
        inject_interval = cfg.inject_seed_interval * (2 ** rep)
        opt = torch.optim.Adam(list(model.parameters()) + list(siren.parameters()), lr=cfg.lr)
        sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=list(cfg.lr_decay_steps), gamma=cfg.lr_decay_gamma)
        scaler = make_grad_scaler(device, precision)
        opt.zero_grad(set_to_none=True)
        acc = 0

        pbar = trange(cfg.train_steps * accumulation_steps, desc=f'Repetition {rep + 1}/{cfg.num_repetitions}')
        for epoch in pbar:
            log_step = (epoch + rep * cfg.train_steps * accumulation_steps) // accumulation_steps

            # ── Pool pruning: replace any fully-collapsed entries with fresh seeds ──
            if epoch % (accumulation_steps * 10) == 0:
                n_dead = prune_dead_pool(pool, model, grid_size)

            idx = torch.randperm(cfg.pool_size, device=device)[:cfg.batch_size]
            x = pool[idx].clone()
            if log_step % inject_interval == 0 and acc == 0:
                x[:1] = model.seed(1, *grid_size)

            n_steps = random.randrange(*cfg.step_range)
            z = None
            with autocast_context(device, precision):
                for _ in range(n_steps):
                    x, z = model(x)

            with torch.no_grad():
                pool[idx] = x.detach()

            generated, x_up = decode_voxel(model, siren, x, z, grid_size, cfg.scale_factor, precision)
            loss, logs = voxel_morphogenesis_loss(
                generated, target, x, x_up, cfg.overflow_weight, cfg.use_sparse_target_loss,
                cfg.sparse_shape_weight, cfg.sparse_foreground_weight,
                cfg.sparse_background_weight, cfg.sparse_state_weight)
            pbar.set_postfix(loss=loss.item())
            loss_to_backprop = loss / accumulation_steps
            if precision == torch.float16:
                scaler.scale(loss_to_backprop).backward()
            else:
                loss_to_backprop.backward()
            acc += 1

            if acc == accumulation_steps:
                if precision == torch.float16:
                    scaler.unscale_(opt)
                normalize_model_grads(model)
                torch.nn.utils.clip_grad_norm_(siren.parameters(), 1.0)
                optimizer_scheduler_step(opt, sched, scaler, precision)
                opt.zero_grad(set_to_none=True)
                acc = 0

            if log_step % 50 == 0 or log_step == cfg.train_steps * cfg.num_repetitions - 1:
                logs['step'] = log_step
                logs['rep'] = rep + 1
                logs['rollout_steps'] = n_steps
                logs['pool_dead'] = n_dead if 'n_dead' in dir() else 0
                history.append(logs)

        with torch.no_grad():
            preview = model.seed(1, *grid_size)
            preview_z = None
            with autocast_context(device, precision):
                for _ in range(max(cfg.step_range) * 2):
                    preview, preview_z = model(preview)
            preview_generated, _ = decode_voxel(model, siren, preview, preview_z, grid_size, cfg.scale_factor, precision)
        show_slices(preview_generated, target, title=f'after repetition {rep + 1}/{cfg.num_repetitions}')
    return model, siren, history


@torch.no_grad()
def rollout(model, siren, cfg, steps=96):
    precision = getattr(model, 'precision', precision_from_cfg(cfg))
    grid_size = tuple(s // cfg.scale_factor for s in target.shape[1:4])
    x = model.seed(1, *grid_size)
    z = None
    with autocast_context(device, precision):
        for _ in range(steps):
            x, z = model(x)
    generated, x_up = decode_voxel(model, siren, x, z, grid_size, cfg.scale_factor, precision)
    return generated, x



## Choose a training target

Use the dropdown in the next cell for the built-in Fall Tree or Pine Tree targets. For your own data, choose a custom mode and fill in the visible path fields?Google Drive paths work after mounting Drive at the top.

`0` means ?preserve the source resolution.?


In [ ]:
#@title Select the training target
#@markdown Pick a built-in target, or choose a custom mode and fill in its path fields below.
TARGET_PRESET = 'Fall Tree (download)' #@param ['Fall Tree (download)', 'Pine Tree (download)', 'Custom dense .vox / .npy', 'Custom OBJ + VOL']
CUSTOM_DENSE_PATH = '' #@param {type:"string"}
CUSTOM_MESH_PATH = '' #@param {type:"string"}
CUSTOM_VOL_PATH = '' #@param {type:"string"}
DENSE_RESOLUTION = 0 #@param {type:"integer"}
DENSE_PADDING = 4 #@param {type:"integer"}
DENSE_MAKE_CUBIC = True #@param {type:"boolean"}
NPY_COORD_ORDER = 'zyx' #@param ['zyx', 'xyz']
MESH_RESOLUTION = 128 #@param {type:"integer"}

PRESET_DENSE_TARGETS = {
    'Fall Tree (download)': (
        'Fall_tree.vox',
        'https://voxbox.store/api/model/download?id=rC2Fnv8YI2',
    ),
    'Pine Tree (download)': (
        'Pine_tree.vox',
        'https://voxbox.store/api/model/download?id=XHxA65ZDGj',
    ),
}

def download_target(filename, url):
    destination = Path(filename)
    if destination.is_file():
        print(f'Using existing target: {destination}')
    else:
        print(f'Downloading target: {destination}')
        urlretrieve(url, destination)
    return destination

DENSE_TARGET_PATH = None
MESH_PATH = None
VOL_PATH = None
DENSE_RESIZE_TO = int(DENSE_RESOLUTION) if int(DENSE_RESOLUTION) > 0 else None

if TARGET_PRESET in PRESET_DENSE_TARGETS:
    filename, url = PRESET_DENSE_TARGETS[TARGET_PRESET]
    DENSE_TARGET_PATH = str(download_target(filename, url))
elif TARGET_PRESET == 'Custom dense .vox / .npy':
    if not CUSTOM_DENSE_PATH.strip():
        raise ValueError('Enter CUSTOM_DENSE_PATH for the custom dense target mode.')
    DENSE_TARGET_PATH = CUSTOM_DENSE_PATH.strip()
elif TARGET_PRESET == 'Custom OBJ + VOL':
    if not CUSTOM_MESH_PATH.strip() or not CUSTOM_VOL_PATH.strip():
        raise ValueError('Enter both CUSTOM_MESH_PATH and CUSTOM_VOL_PATH for OBJ + VOL mode.')
    MESH_PATH = CUSTOM_MESH_PATH.strip()
    VOL_PATH = CUSTOM_VOL_PATH.strip()
else:
    raise ValueError(f'Unknown target preset: {TARGET_PRESET}')

if DENSE_TARGET_PATH is not None:
    target = dense_file_to_target(
        DENSE_TARGET_PATH,
        res=DENSE_RESIZE_TO,
        padding=int(DENSE_PADDING),
        coord_order=NPY_COORD_ORDER,
        make_cubic=bool(DENSE_MAKE_CUBIC),
        ensure_divisible_by=cfg.scale_factor,
    )
    target_path_for_export = DENSE_TARGET_PATH
else:
    target = load_mesh_solid_texture_target(
        MESH_PATH,
        VOL_PATH,
        res=int(MESH_RESOLUTION),
        padding=int(DENSE_PADDING),
        ensure_divisible_by=cfg.scale_factor,
    )
    target_path_for_export = MESH_PATH

spatial = tuple(int(s) for s in target.shape[1:4])
if any(s % cfg.scale_factor != 0 for s in spatial):
    raise ValueError(f'target shape {spatial} must be divisible by scale_factor={cfg.scale_factor}')

print('target preset:', TARGET_PRESET)
print('target_path_for_export:', target_path_for_export)
print('target:', tuple(target.shape), 'NCA grid:', tuple(s // cfg.scale_factor for s in spatial))


In [ ]:
def to_np_rgba(v):
    if isinstance(v, torch.Tensor):
        v = v.detach().cpu().numpy()
    if v.ndim == 5:  # [B,D,H,W,4]
        v = v[0]
    return v

def make_coarse_rgba(vol, scale_factor=2):
    """
    vol: [D,H,W,4] in Z,Y,X,RGBA
    returns coarse [D/s,H/s,W/s,4]
    """
    D, H, W, C = vol.shape
    s = scale_factor

    D2 = (D // s) * s
    H2 = (H // s) * s
    W2 = (W // s) * s
    vol = vol[:D2, :H2, :W2]

    rgba = torch.from_numpy(vol).float().permute(3, 0, 1, 2).unsqueeze(0)  # [1,4,D,H,W]

    alpha = rgba[:, 3:4]
    coarse_alpha = F.max_pool3d(alpha, kernel_size=s, stride=s)

    rgb = rgba[:, :3] * alpha
    coarse_rgb_sum = F.avg_pool3d(rgb, kernel_size=s, stride=s) * (s ** 3)
    coarse_count = F.avg_pool3d(alpha, kernel_size=s, stride=s) * (s ** 3)
    coarse_rgb = coarse_rgb_sum / coarse_count.clamp_min(1e-6)

    coarse = torch.cat([coarse_rgb, coarse_alpha], dim=1)
    coarse = coarse[0].permute(1, 2, 3, 0).numpy()  # [D,H,W,4]
    return coarse

def show_voxels_matplotlib(vol, threshold=0.5, title="voxel"):
    """
    vol: [D,H,W,4] in Z,Y,X,RGBA
    """
    alpha = vol[..., 3]
    filled = alpha > threshold

    colors = vol.copy()
    colors[..., 3] = np.where(filled, 1.0, 0.0)

    # matplotlib wants X,Y,Z
    filled_xyz = np.transpose(filled, (2, 1, 0))
    colors_xyz = np.transpose(colors, (2, 1, 0, 3))

    fig = plt.figure(figsize=(9, 9))
    ax = fig.add_subplot(projection="3d")
    ax.voxels(filled_xyz, facecolors=colors_xyz, edgecolor=None)
    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_box_aspect(filled_xyz.shape)
    plt.tight_layout()
    plt.show()

vol = to_np_rgba(target)
coarse = make_coarse_rgba(vol, scale_factor=cfg.scale_factor)

print("full:", vol.shape)
print("coarse:", coarse.shape)
print("coarse occupancy fraction:", coarse[..., 3].mean())
print("coarse occupied voxels:", int((coarse[..., 3] > 0.5).sum()))

show_voxels_matplotlib(coarse, threshold=0.5, title=f"coarse target, scale={cfg.scale_factor}")

## Train

The settings live in `CFG` above. Training uses a state pool, seed reinjection, stochastic NCA updates, Adam with learning-rate decay, normalized NCA gradients, and the selected voxel-loss mode.

For a smoke test, lower `cfg.train_steps`. For a final model, keep the target and configuration fixed, train on GPU, then judge the actual generated rollout rather than only the loss curve.


In [ ]:

model, siren, history = train_voxel_gnca(target, cfg)
plot_history(history)

generated, final_state = rollout(model, siren, cfg, steps=max(cfg.step_range) * 2)
show_slices(generated, target, title='voxel morphogenesis rollout')
voxel_scatter(generated, threshold=0.45)


## Optional recovery probe and local checkpoint

The recovery probe checks whether the pool-trained attractor survives a center cut; it is not separate damage training. The following checkpoint cell is optional because the verified Google Drive backup at the end also saves model weights, configuration, and history.


In [ ]:

@torch.no_grad()
def damage_state_center(state, frac=0.35):
    x = state.clone()
    _, _, d, h, w = x.shape
    rd, rh, rw = int(d * frac / 2), int(h * frac / 2), int(w * frac / 2)
    cd, ch, cw = d // 2, h // 2, w // 2
    x[:, :, cd-rd:cd+rd, ch-rh:ch+rh, cw-rw:cw+rw] = 0
    return x

@torch.no_grad()
def repair_rollout(model, siren, state, cfg, repair_steps=64):
    precision = getattr(model, 'precision', precision_from_cfg(cfg))
    grid_size = tuple(s // cfg.scale_factor for s in target.shape[1:4])
    x = damage_state_center(state)
    z = None
    with autocast_context(device, precision):
        for _ in range(repair_steps):
            x, z = model(x)
    return decode_voxel(model, siren, x, z, grid_size, cfg.scale_factor, precision)[0]

repaired = repair_rollout(model, siren, final_state, cfg)
show_slices(repaired, target, title='after center damage + repair rollout')


In [ ]:

ckpt_path = Path('gnca_3d_voxel_morphogenesis.pt')
torch.save({'cfg': cfg.__dict__, 'model': model.state_dict(), 'siren': siren.state_dict(), 'history': history}, ckpt_path)
print('saved', ckpt_path.resolve())


## Export for the WebGPU voxel demo

Run the code cell immediately below **after training**. It writes the three files used by `demo/growing_voxels/`:

- `nca_manifest.json`
- `nca_weights.bin`
- `nca_weights.json` (human-readable debug copy)

Default local output folder: `./nca_export`


In [ ]:
class ExportableVoxelNCA:
    def __init__(self, model, siren, cfg, target=None, target_path=None, iterations=0):
        self.model = model
        self.lppn = siren
        self.channels = int(model.channels)
        self.scale = float(cfg.scale_factor)
        self.living_channel = 3
        self.living_threshold = float(getattr(cfg, 'living_threshold', 0.1))
        self.edge_loss_weight = 0.0
        self.iterations = int(iterations)
        self.target_path = target_path

        if target is not None:
            spatial = tuple(int(v) for v in target.shape[1:4])
            if len(set(spatial)) != 1:
                raise ValueError(f'Exporter manifest expects cubic target, got {spatial}')
            self.target_size = spatial[0]
            self.coarse_size = self.target_size // int(cfg.scale_factor)

            # Calculate actual occupancy of the target and add some headroom
            # target shape is assumed to be [C, S, S, S] where C >= 4 (RGBA)
            # or [S, S, S] binary mask.
            try:
                if len(target.shape) == 4 and target.shape[0] >= 4:
                    alpha = target[3]
                else:
                    alpha = target
                filled = float((alpha > 0.0).float().sum().item())
                total = float(alpha.numel())
                # Add 5% occupancy buffer headroom, max out at 1.0, min at 0.1
                self.max_occupancy = min(1.0, max(0.1, (filled / total) + 0.05))
            except Exception as e:
                self.max_occupancy = float(getattr(cfg, "max_occupancy", 0.3))
        else:
            self.target_size = int(cfg.target_inner_res + 2 * cfg.padding)
            self.coarse_size = self.target_size // int(cfg.scale_factor)
            self.max_occupancy = float(getattr(cfg, "max_occupancy", 0.3))

    def state_dict(self):
        sd = OrderedDict()
        sd['perception.perceive.weight'] = self.model.filters[:, None].detach()
        sd['adaptation.adapt.0.weight'] = self.model.w1.weight.detach()
        sd['adaptation.adapt.0.bias'] = self.model.w1.bias.detach()
        sd['adaptation.adapt.2.weight'] = self.model.w2.weight.detach()
        if self.model.w2.bias is not None:
            sd['adaptation.adapt.2.bias'] = self.model.w2.bias.detach()
        for name, tensor in self.lppn.state_dict().items():
            sd[f'lppn.{name}'] = tensor.detach()
        return sd


def export_nca_weights(
    nca,
    iterations: int = 0,
    folder_path: str = "./nca_export",
    json_filename: str = "nca_weights.json",
    bin_filename: str = "nca_weights.bin",
    manifest_filename: str = "nca_manifest.json",
):
    """
    Export weights for this app's loader.

    The app loads:
      - nca_manifest.json
      - nca_weights.bin

    Manifest format:
    {
      "meta": { "channels": ..., "coarse_size": ..., "scale": ... },
      "perception": { "perceive.weight": { "offset": BYTE_OFFSET, "count": FLOAT_COUNT } },
      "adaptation": { "adapt.0.weight": { "offset": BYTE_OFFSET, "count": FLOAT_COUNT } },
      "lppn": { "net.0.linear.weight": { "offset": BYTE_OFFSET, "count": FLOAT_COUNT } }
    }

    Important: offset is in BYTES, not float32 elements.
    """

    folder = Path(folder_path)
    folder.mkdir(parents=True, exist_ok=True)

    json_path = folder / json_filename
    bin_path = folder / bin_filename
    manifest_path = folder / manifest_filename

    state_dict = nca.state_dict()

    manifest = {
        "meta": {
            "channels": int(nca.channels),
            "coarse_size": int(nca.coarse_size),
            "target_size": int(nca.target_size) if nca.target_size is not None else None,
            "scale": float(nca.scale) if nca.scale is not None else None,
            "living_channel": int(getattr(nca, "living_channel", 0)),
            "living_threshold": float(getattr(nca, "living_threshold", 0.1)),
            "num_frequencies": int(getattr(nca.lppn, "num_frequencies", 1)),
            "lppn_first_omega_0": float(getattr(nca.lppn.net[0], "omega_0", 10.0)),
            "lppn_hidden_omega_0": float(getattr(nca.lppn.net[1], "omega_0", 10.0)),
            "edge_loss_weight": float(getattr(nca, "edge_loss_weight", 0.0)),
            "iterations": int(iterations),
            "max_occupancy": float(getattr(nca, "max_occupancy", 0.3)),
            "gpu_used": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "target_path": getattr(nca, "target_path", None),
        },
        "perception": {},
        "adaptation": {},
        "lppn": {},
    }

    if psutil is not None:
        manifest["meta"]["ram_used_gb"] = round(
            psutil.virtual_memory().used / (1024 ** 3), 2
        )

    byte_offset = 0

    with open(bin_path, "wb") as bf:
        for full_name, tensor in state_dict.items():
            arr = tensor.detach().cpu().contiguous().numpy().astype(np.float32)
            flat = arr.reshape(-1)

            if full_name.startswith("perception."):
                section = "perception"
                short_name = full_name.replace("perception.", "", 1)
            elif full_name.startswith("adaptation."):
                section = "adaptation"
                short_name = full_name.replace("adaptation.", "", 1)
            elif full_name.startswith("lppn."):
                section = "lppn"
                short_name = full_name.replace("lppn.", "", 1)
            else:
                print(f"Skipping non-exported tensor: {full_name}")
                continue

            bf.write(flat.tobytes())

            manifest[section][short_name] = {
                "offset": int(byte_offset),
                "count": int(flat.size),
                "shape": list(arr.shape),
            }

            byte_offset += int(flat.nbytes)

    with open(manifest_path, "w") as mf:
        json.dump(manifest, mf, indent=2)

    # Optional human/debug copy. The app does not load this.
    with open(json_path, "w") as jf:
        json.dump(manifest, jf, indent=2)

    print(f"Exported runtime manifest : {manifest_path}")
    print(f"Exported binary file      : {bin_path}")
    print(f"Exported JSON copy        : {json_path}")
    print()
    print("Groups exported:")
    print(f"  perception: {len(manifest['perception'])} tensors")
    print(f"  adaptation: {len(manifest['adaptation'])} tensors")
    print(f"  lppn      : {len(manifest['lppn'])} tensors")
    print()
    print(f"Total float32 values: {byte_offset // 4:,}")
    print(f"Total bytes         : {byte_offset:,}")

    return json_path, bin_path, manifest_path


# Re-export only. No retraining needed.
nca = ExportableVoxelNCA(
    model,
    siren,
    cfg,
    target=target,
    target_path=target_path_for_export,
    iterations=cfg.train_steps,
)
SAVE_PATH = "./nca_export"

export_nca_weights(
    nca,
    iterations=nca.iterations,
    folder_path=SAVE_PATH,
    json_filename="nca_weights.json",
    bin_filename="nca_weights.bin",
    manifest_filename="nca_manifest.json",
)



## Google Drive backup and safe shutdown

Run the backup cell after training and the export cell, then run the shutdown cell only after it prints `BACKUP COMPLETE AND VERIFIED`. Each backup is timestamped, so it never overwrites an earlier run.

The backup contains the current NCA/SIREN weights, configuration, history, target file when available, and a freshly exported, validated web-model package. It is a weights checkpoint rather than an exact mid-training resume checkpoint: the current training loop does not retain its optimizer, scheduler, GradScaler, or state pool after it returns.


In [ ]:
#@title 1. Back up the current training run to Google Drive
#@markdown Run this **after training** and after the exporter cell above. It creates a fresh, validated web export and a portable weights checkpoint.

required = ('model', 'siren', 'cfg', 'history', 'target', 'target_path_for_export',
            'ExportableVoxelNCA', 'export_nca_weights')
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        'Run training and the FINAL EXPORT CELL first; missing: ' + ', '.join(missing)
    )

if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive', force_remount=False)

RUN_LABEL = Path(str(target_path_for_export)).stem.replace(' ', '_') or 'voxel_run'
DRIVE_ROOT = Path('/content/drive/MyDrive/Cells2Voxels_backups')
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
snapshot_name = f'{RUN_LABEL}_{timestamp}'
local_snapshot = Path('/content/gnca_backups') / snapshot_name
local_web_export = local_snapshot / 'web_model'
drive_snapshot = DRIVE_ROOT / snapshot_name

if local_snapshot.exists() or drive_snapshot.exists():
    raise FileExistsError(f'Refusing to overwrite an existing backup: {snapshot_name}')
local_web_export.mkdir(parents=True, exist_ok=False)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

def cpu_state_dict(module):
    return {
        key: value.detach().cpu().contiguous().clone()
        for key, value in module.state_dict().items()
    }

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

cfg_dict = asdict(cfg) if is_dataclass(cfg) else dict(vars(cfg))
completed_updates = (
    int(history[-1]['step']) + 1
    if history and 'step' in history[-1]
    else int(cfg.train_steps * cfg.num_repetitions)
)

# Portable checkpoint: model weights, decoder weights, configuration, and metrics.
checkpoint = {
    'format_version': 1,
    'saved_at_utc': timestamp,
    'completed_updates': completed_updates,
    'cfg': cfg_dict,
    'model': cpu_state_dict(model),
    'siren': cpu_state_dict(siren),
    'history': history,
    'target_path_for_export': target_path_for_export,
}
torch.save(checkpoint, local_snapshot / 'training_checkpoint.pt')
(local_snapshot / 'cfg.json').write_text(json.dumps(cfg_dict, indent=2), encoding='utf-8')
(local_snapshot / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')

# Export fresh files from the in-memory model; do not reuse a possibly stale local export.
nca_backup = ExportableVoxelNCA(
    model,
    siren,
    cfg,
    target=target,
    target_path=target_path_for_export,
    iterations=completed_updates,
)
json_path, bin_path, manifest_path = export_nca_weights(
    nca_backup,
    iterations=completed_updates,
    folder_path=str(local_web_export),
    json_filename='nca_weights.json',
    bin_filename='nca_weights.bin',
    manifest_filename='nca_manifest.json',
)

# Validate that the binary actually contains every tensor the manifest declares.
manifest = json.loads(Path(manifest_path).read_text(encoding='utf-8'))
expected_bin_bytes = max(
    item['offset'] + item['count'] * 4
    for group in ('perception', 'adaptation', 'lppn')
    for item in manifest[group].values()
)
actual_bin_bytes = Path(bin_path).stat().st_size
if actual_bin_bytes != expected_bin_bytes:
    raise RuntimeError(
        f'Invalid web export: binary has {actual_bin_bytes:,} bytes; '
        f'manifest requires {expected_bin_bytes:,}.'
    )

# Preserve whichever source files define the selected target when available.
source_paths = (
    globals().get('DENSE_TARGET_PATH'),
    globals().get('MESH_PATH'),
    globals().get('VOL_PATH'),
)
copied_sources = set()
for source_path in source_paths:
    if not source_path:
        continue
    source_file = Path(str(source_path))
    if source_file.is_file() and source_file.name not in copied_sources:
        shutil.copy2(source_file, local_snapshot / source_file.name)
        copied_sources.add(source_file.name)

# Copy the complete local snapshot to Drive, then verify every file byte-for-byte.
shutil.copytree(local_snapshot, drive_snapshot)
for source in local_snapshot.rglob('*'):
    if source.is_file():
        copied = drive_snapshot / source.relative_to(local_snapshot)
        if not copied.is_file() or copied.stat().st_size != source.stat().st_size:
            raise RuntimeError(f'Drive copy failed size check: {copied}')
        if sha256(copied) != sha256(source):
            raise RuntimeError(f'Drive copy failed hash check: {copied}')

(drive_snapshot / 'BACKUP_COMPLETE.txt').write_text(
    f'Verified UTC backup: {timestamp}\nWeb binary bytes: {actual_bin_bytes}\n',
    encoding='utf-8',
)
if hasattr(os, 'sync'):
    os.sync()

DRIVE_BACKUP_COMPLETE = True
DRIVE_BACKUP_PATH = str(drive_snapshot)
print(f'BACKUP COMPLETE AND VERIFIED:\n{DRIVE_BACKUP_PATH}')


In [ ]:
#@title 2. Shut down this Colab runtime after the verified backup
#@markdown This disconnects and unassigns the runtime. Run it only after the backup cell prints `BACKUP COMPLETE AND VERIFIED`.

if not globals().get('DRIVE_BACKUP_COMPLETE', False):
    raise RuntimeError('No verified Drive backup exists in this session; refusing to shut down.')
if not (Path(DRIVE_BACKUP_PATH) / 'BACKUP_COMPLETE.txt').is_file():
    raise RuntimeError('Backup completion marker is missing; refusing to shut down.')

print(f'Flushing verified backup: {DRIVE_BACKUP_PATH}')
drive.flush_and_unmount(timeout_ms=120_000)
print('Drive flushed. Unassigning the Colab runtime now...')
runtime.unassign()
